In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Project root:", project_root)
print("Raw data exists:", raw_data_dir.exists())

Python: 3.14.3
Pandas: 3.0.5
Project root: c:\Users\miang\OneDrive\Documents\Github\freddie-mac-credit-risk
Raw data exists: True


# Freddie Mac Credit Risk: Data Audit

## Project purpose

This project examines mortgage credit risk across different economic environments.

### Baseline PD model
- Vintages: 2015–2017
- Training data: 2015–2016
- Test data: 2017
- Preliminary outcome: whether a loan reaches 90+ days delinquent within its first 24 months
- Predictors: origination-time information only

### Vintage stress comparison
- Compare the 2006 and 2016 vintages
- Measure performance at equal loan ages through 36 months
- Examine delinquency, cure rates, transitions, survival, and borrower-risk segments

## Data-audit objectives
1. Confirm all required files exist.
2. Verify row and column counts.
3. Inspect the pipe-delimited structure.
4. Check whether file layouts differ across vintages.
5. Confirm loan identifiers can connect origination and monthly-performance records.

In [2]:
vintages = [2006, 2015, 2016, 2017]

origination_files = {
    year: raw_data_dir / str(year) / f"sample_orig_{year}.txt"
    for year in vintages
}

audit_results = []

for year, file_path in origination_files.items():
    sample = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        dtype="string",
        nrows=5
    )

    with file_path.open("r", encoding="utf-8") as file:
        row_count = sum(1 for _ in file)

    audit_results.append({
        "vintage": year,
        "file_exists": file_path.exists(),
        "file_size_mb": round(file_path.stat().st_size / (1024**2), 2),
        "row_count": row_count,
        "column_count": sample.shape[1]
    })

origination_audit = pd.DataFrame(audit_results)
origination_audit

,vintage,file_exists,file_size_mb,row_count,column_count
0,2006,True,6.11,50000,31
1,2015,True,5.95,50000,31
2,2016,True,5.98,50000,31
3,2017,True,6.00,50000,31


In [3]:
orig_2015_preview = pd.read_csv(
    origination_files[2015],
    sep="|",
    header=None,
    dtype="string",
    nrows=3
)

orig_2015_preview.columns = [
    f"field_{number:02d}"
    for number in range(1, orig_2015_preview.shape[1] + 1)
]

orig_2015_preview.T

,0,1,2
field_01,734,797,670
field_02,201504,201504,201504
field_03,N,N,N
field_04,203003,204503,203003
field_05,<NA>,<NA>,11460
field_06,0,0,12
field_07,1,1,1
field_08,P,S,P
field_09,80,37,90
field_10,15,26,33


In [4]:
orig_2015_ids = pd.read_csv(
    origination_files[2015],
    sep="|",
    header=None,
    usecols=[19],
    names=["loan_sequence_number"],
    dtype="string"
)

performance_2015_file = (
    raw_data_dir / "2015" / "sample_perf_2015.txt"
)

perf_2015_sample = pd.read_csv(
    performance_2015_file,
    sep="|",
    header=None,
    usecols=[0],
    names=["loan_sequence_number"],
    dtype="string",
    nrows=100_000
)

orig_id_set = set(orig_2015_ids["loan_sequence_number"])

id_check = pd.Series({
    "origination_rows": len(orig_2015_ids),
    "unique_origination_ids": orig_2015_ids["loan_sequence_number"].nunique(),
    "missing_origination_ids": orig_2015_ids["loan_sequence_number"].isna().sum(),
    "performance_rows_tested": len(perf_2015_sample),
    "performance_unique_ids_tested": perf_2015_sample["loan_sequence_number"].nunique(),
    "performance_ids_found_in_origination": (
        perf_2015_sample["loan_sequence_number"].isin(orig_id_set).sum()
    ),
    "performance_ids_not_found": (
        ~perf_2015_sample["loan_sequence_number"].isin(orig_id_set)
    ).sum()
})

id_check

origination_rows                         50000
unique_origination_ids                   50000
missing_origination_ids                      0
performance_rows_tested                 100000
performance_unique_ids_tested             1458
performance_ids_found_in_origination    100000
performance_ids_not_found                    0
dtype: int64

## Official origination-file schema

The July 2026 Freddie Mac sample origination files contain 31 fields.  
The project uses readable snake_case names while preserving the official field order.

In [5]:
origination_columns = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_indicator",
    "maturity_date",
    "msa",
    "mi_percentage",
    "number_of_units",
    "occupancy_status",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "channel",
    "prepayment_penalty_indicator",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_identifier",
    "loan_purpose",
    "original_loan_term",
    "number_of_borrowers",
    "seller_name",
    "super_conforming_flag",
    "pre_harp_loan_identifier",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "vantagescore_4"
]

assert len(origination_columns) == 31

origination_frames = []

for year, file_path in origination_files.items():
    vintage_data = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        names=origination_columns,
        dtype="string",
        na_filter=False
    )

    vintage_data.insert(0, "vintage", year)
    origination_frames.append(vintage_data)

originations_raw = pd.concat(
    origination_frames,
    ignore_index=True
)

schema_check = pd.Series({
    "total_rows": len(originations_raw),
    "total_columns_including_vintage": originations_raw.shape[1],
    "unique_loan_identifiers": originations_raw["loan_identifier"].nunique(),
    "blank_loan_identifiers": (
        originations_raw["loan_identifier"].str.strip().eq("").sum()
    )
})

schema_check

total_rows                         200000
total_columns_including_vintage        32
unique_loan_identifiers            200000
blank_loan_identifiers                  0
dtype: int64

In [6]:
vantage_numeric = pd.to_numeric(
    originations_raw["vantagescore_4"],
    errors="coerce"
)

vantage_schema_check = (
    pd.DataFrame({
        "vintage": originations_raw["vintage"],
        "available_300_to_850": vantage_numeric.between(300, 850),
        "not_available_9999": vantage_numeric.eq(9999),
        "blank": originations_raw["vantagescore_4"].str.strip().eq("")
    })
    .groupby("vintage")
    .sum()
)

vantage_schema_check

,available_300_to_850,not_available_9999,blank
vintage,,,
2006,0,50000,0
2015,0,50000,0
2016,0,50000,0
2017,0,50000,0


In [7]:
official_missing_codes = {
    "credit_score": {"9999"},
    "first_time_homebuyer_indicator": {"9"},
    "msa": set(),
    "mi_percentage": {"999"},
    "number_of_units": {"99"},
    "occupancy_status": {"9"},
    "original_cltv": {"999"},
    "original_dti": {"999"},
    "original_ltv": {"999"},
    "channel": {"9"},
    "property_type": {"99"},
    "postal_code": {"000"},
    "loan_purpose": {"9"},
    "number_of_borrowers": {"99"},
    "pre_harp_loan_identifier": set(),
    "special_eligibility_program": set(),
    "property_valuation_method": {"7"},
    "vantagescore_4": {"9999"}
}

missing_audit_rows = []

for column in origination_columns:
    values = originations_raw[column].str.strip()
    missing_codes = official_missing_codes.get(column, set())

    blank_count = values.eq("").sum()
    sentinel_count = values.isin(missing_codes).sum()

    missing_audit_rows.append({
        "column": column,
        "unique_raw_values": values.nunique(),
        "blank_count": blank_count,
        "sentinel_count": sentinel_count,
        "total_missing_or_sentinel": blank_count + sentinel_count,
        "missing_percent": round(
            100 * (blank_count + sentinel_count) / len(values),
            2
        )
    })

missing_value_audit = (
    pd.DataFrame(missing_audit_rows)
    .sort_values("missing_percent", ascending=False)
    .reset_index(drop=True)
)

missing_value_audit

,column,unique_raw_values,blank_count,sentinel_count,total_missing_or_sentinel,missing_percent
0,vantagescore_4,1,0,200000,200000,100.00
1,property_valuation_method,2,0,199555,199555,99.78
2,special_eligibility_program,3,195128,0,195128,97.56
3,pre_harp_loan_identifier,8347,191654,0,191654,95.83
4,msa,458,24620,0,24620,12.31
5,original_dti,66,0,9423,9423,4.71
6,credit_score,353,0,54,54,0.03
7,first_time_homebuyer_indicator,3,0,23,23,0.01
8,number_of_borrowers,3,0,19,19,0.01
9,original_cltv,193,0,10,10,0.00


## Initial data-quality decisions

- The four origination files contain 200,000 unique loans with no missing identifiers.
- `vantagescore_4` is unavailable for every sampled loan and will be excluded from modeling.
- `property_valuation_method` is nearly entirely unavailable and will be excluded from the primary PD model.
- Blank `pre_harp_loan_identifier` values are structurally not applicable for most loans, rather than ordinary missing data.
- Blank `special_eligibility_program` values mean not available or not applicable.
- Missing-value codes in the remaining predictors will be converted to proper null values.
- Variables will not be removed solely because they contain some missing data; modeling treatment will be decided later.

In [8]:
sparse_columns = [
    "vantagescore_4",
    "property_valuation_method",
    "special_eligibility_program",
    "pre_harp_loan_identifier",
    "msa",
    "original_dti",
    "credit_score"
]

missing_by_vintage_rows = []

for year in vintages:
    year_data = originations_raw.loc[
        originations_raw["vintage"].eq(year)
    ]

    for column in sparse_columns:
        values = year_data[column].str.strip()
        missing_codes = official_missing_codes.get(column, set())

        missing_count = (
            values.eq("") | values.isin(missing_codes)
        ).sum()

        missing_by_vintage_rows.append({
            "vintage": year,
            "column": column,
            "missing_count": missing_count,
            "missing_percent": round(
                100 * missing_count / len(year_data),
                2
            )
        })

missing_by_vintage = (
    pd.DataFrame(missing_by_vintage_rows)
    .pivot(
        index="column",
        columns="vintage",
        values="missing_percent"
    )
)

missing_by_vintage

vintage,2006,2015,2016,2017
column,,,,
credit_score,0.09,0.00,0.00,0.02
msa,17.12,11.21,10.11,10.80
original_dti,2.14,8.04,5.06,3.61
pre_harp_loan_identifier,100.00,91.97,94.95,96.39
property_valuation_method,100.00,100.00,100.00,99.11
special_eligibility_program,100.00,99.35,97.88,93.02
vantagescore_4,100.00,100.00,100.00,100.00


## Clean the origination data

The raw dataframe remains unchanged. A separate cleaned dataframe converts blank
strings and Freddie Mac missing-value codes into proper null values. Numeric and
date fields are then assigned appropriate data types.

In [9]:
originations_clean = originations_raw.copy()

# Convert blank strings and official sentinel codes to proper null values
for column in origination_columns:
    values = originations_clean[column].str.strip()
    values = values.mask(values.eq(""), pd.NA)

    missing_codes = official_missing_codes.get(column, set())
    if missing_codes:
        values = values.mask(values.isin(missing_codes), pd.NA)

    originations_clean[column] = values

numeric_columns = [
    "credit_score",
    "mi_percentage",
    "number_of_units",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers",
    "property_valuation_method",
    "vantagescore_4"
]

for column in numeric_columns:
    originations_clean[column] = pd.to_numeric(
        originations_clean[column],
        errors="coerce"
    )

date_columns = [
    "first_payment_date",
    "maturity_date"
]

for column in date_columns:
    originations_clean[column] = pd.to_datetime(
        originations_clean[column],
        format="%Y%m",
        errors="coerce"
    )

originations_clean.head()

,vintage,credit_score,first_payment_date,first_time_homebuyer_indicator,maturity_date,msa,mi_percentage,number_of_units,occupancy_status,original_cltv,...,original_loan_term,number_of_borrowers,seller_name,super_conforming_flag,pre_harp_loan_identifier,special_eligibility_program,harp_indicator,property_valuation_method,interest_only_indicator,vantagescore_4
0,2006,776,2006-03-01,N,2036-02-01,<NA>,12,1,P,82,...,360,2,OTHER,N,<NA>,<NA>,N,<NA>,N,<NA>
1,2006,639,2006-03-01,N,2036-02-01,39300,12,1,P,81,...,360,2,OTHER,N,<NA>,<NA>,N,<NA>,N,<NA>
2,2006,707,2006-03-01,N,2036-02-01,<NA>,12,1,P,82,...,360,2,OTHER,N,<NA>,<NA>,N,<NA>,N,<NA>
3,2006,722,2006-03-01,N,2021-02-01,10580,0,1,P,72,...,180,2,OTHER,N,<NA>,<NA>,N,<NA>,N,<NA>
4,2006,772,2006-04-01,N,2036-03-01,37380,0,1,P,62,...,360,1,OTHER,N,<NA>,<NA>,N,<NA>,N,<NA>


In [10]:
primary_model_exclusions = [
    "loan_identifier",
    "pre_harp_loan_identifier",
    "vantagescore_4",
    "property_valuation_method",
    "special_eligibility_program"
]

cleaning_check = pd.Series({
    "raw_rows": len(originations_raw),
    "clean_rows": len(originations_clean),
    "unique_clean_loan_ids": originations_clean["loan_identifier"].nunique(),
    "blank_clean_loan_ids": originations_clean["loan_identifier"].isna().sum(),
    "duplicate_clean_loan_ids": originations_clean["loan_identifier"].duplicated().sum(),
    "excluded_from_primary_model": len(primary_model_exclusions)
})

cleaning_check

raw_rows                       200000
clean_rows                     200000
unique_clean_loan_ids          200000
blank_clean_loan_ids                0
duplicate_clean_loan_ids            0
excluded_from_primary_model         5
dtype: int64

In [11]:
clean_missing_check = (
    originations_clean
    .groupby("vintage")[
        [
            "credit_score",
            "msa",
            "original_dti",
            "pre_harp_loan_identifier",
            "property_valuation_method",
            "special_eligibility_program",
            "vantagescore_4"
        ]
    ]
    .agg(lambda column: round(100 * column.isna().mean(), 2))
    .T
)

clean_missing_check

vintage,2006,2015,2016,2017
credit_score,0.09,0.0,0.0,0.02
msa,17.12,11.21,10.11,10.8
original_dti,2.14,8.04,5.06,3.61
pre_harp_loan_identifier,100.0,91.97,94.95,96.39
property_valuation_method,100.0,100.0,100.0,99.11
special_eligibility_program,100.0,99.35,97.88,93.02
vantagescore_4,100.0,100.0,100.0,100.0


## Range and category validation

Numeric variables are checked for implausible values after missing-value codes have
been converted to nulls. Categorical fields are inspected for unexpected codes.

In [12]:
numeric_range_rules = {
    "credit_score": (300, 850),
    "mi_percentage": (0, 100),
    "number_of_units": (1, 4),
    "original_cltv": (0, 300),
    "original_dti": (0, 100),
    "original_upb": (1, None),
    "original_ltv": (0, 300),
    "original_interest_rate": (0, 25),
    "original_loan_term": (1, 600),
    "number_of_borrowers": (1, 10)
}

range_check_rows = []

for column, (minimum, maximum) in numeric_range_rules.items():
    values = originations_clean[column]

    invalid = values.notna() & values.lt(minimum)

    if maximum is not None:
        invalid = invalid | (
            values.notna() & values.gt(maximum)
        )

    range_check_rows.append({
        "column": column,
        "nonmissing_count": values.notna().sum(),
        "minimum_observed": values.min(),
        "maximum_observed": values.max(),
        "outside_review_range": invalid.sum()
    })

numeric_range_check = pd.DataFrame(range_check_rows)
numeric_range_check

,column,nonmissing_count,minimum_observed,maximum_observed,outside_review_range
0,credit_score,199946,300.00,844.00,0
1,mi_percentage,200000,0.00,40.00,0
2,number_of_units,200000,1.00,4.00,0
3,original_cltv,199990,5.00,854.00,3
4,original_dti,190577,1.00,65.00,0
5,original_upb,200000,10000.00,1000000.00,0
6,original_ltv,199990,3.00,336.00,1
7,original_interest_rate,200000,2.25,9.79,0
8,original_loan_term,200000,60.00,480.00,0
9,number_of_borrowers,199981,1.00,2.00,0


In [13]:
categorical_columns = [
    "first_time_homebuyer_indicator",
    "occupancy_status",
    "channel",
    "prepayment_penalty_indicator",
    "amortization_type",
    "property_state",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
    "harp_indicator",
    "interest_only_indicator"
]

category_check = pd.DataFrame({
    "column": categorical_columns,
    "nonmissing_unique_values": [
        sorted(
            originations_clean[column]
            .dropna()
            .unique()
            .tolist()
        )
        for column in categorical_columns
    ]
})

category_check

,column,nonmissing_unique_values
0,first_time_homebuyer_indicator,"[N, Y]"
1,occupancy_status,"[I, P, S]"
2,channel,"[B, C, R, T]"
3,prepayment_penalty_indicator,"[N, Y]"
4,amortization_type,[FRM]
5,property_state,"[AK, AL, AR, AZ, CA, CO, CT, DC, DE, FL, GA, G..."
6,property_type,"[CO, CP, MH, PU, SF]"
7,loan_purpose,"[C, N, P]"
8,super_conforming_flag,"[N, Y]"
9,harp_indicator,"[N, Y]"


In [14]:
ratio_review_mask = (
    originations_clean["original_cltv"].gt(300)
    | originations_clean["original_ltv"].gt(300)
)

ratio_review_columns = [
    "vintage",
    "loan_identifier",
    "loan_purpose",
    "harp_indicator",
    "pre_harp_loan_identifier",
    "occupancy_status",
    "property_type",
    "original_upb",
    "original_ltv",
    "original_cltv",
    "credit_score",
    "original_dti"
]

ratio_review = (
    originations_clean
    .loc[ratio_review_mask, ratio_review_columns]
    .copy()
)

ratio_review["cltv_minus_ltv"] = (
    ratio_review["original_cltv"]
    - ratio_review["original_ltv"]
)

ratio_review.sort_values(
    ["original_cltv", "original_ltv"],
    ascending=False
)

,vintage,loan_identifier,loan_purpose,harp_indicator,pre_harp_loan_identifier,occupancy_status,property_type,original_upb,original_ltv,original_cltv,credit_score,original_dti,cltv_minus_ltv
97169,2015,F15Q40257850,N,Y,F03Q30931098,I,PU,78000,45,854,766,<NA>,809
137415,2016,F16Q30476204,N,Y,A08Q30002418,P,CO,138000,336,336,762,<NA>,0
108090,2016,F16Q10200259,N,Y,F07Q10243093,I,CO,125000,260,333,637,<NA>,73


### Review of unusually high mortgage ratios

Three loans exceeded the project's conservative ratio-review threshold. All three
are identified as HARP loans, contain pre-HARP loan identifiers, and have CLTV
greater than or equal to LTV. These characteristics indicate legitimate high-ratio
refinance observations rather than file-loading errors.

The disclosed LTV and CLTV values will therefore remain unchanged. A separate
review flag will identify these observations for later sensitivity analysis,
particularly for models that may be influenced by extreme numeric values.

In [15]:
originations_clean["extreme_ratio_review_flag"] = (
    originations_clean["original_cltv"].gt(300)
    | originations_clean["original_ltv"].gt(300)
)

extreme_ratio_check = pd.Series({
    "flagged_loans": originations_clean[
        "extreme_ratio_review_flag"
    ].sum(),
    "flagged_harp_loans": originations_clean.loc[
        originations_clean["extreme_ratio_review_flag"],
        "harp_indicator"
    ].eq("Y").sum(),
    "flagged_with_pre_harp_id": originations_clean.loc[
        originations_clean["extreme_ratio_review_flag"],
        "pre_harp_loan_identifier"
    ].notna().sum(),
    "flagged_cltv_below_ltv": (
        originations_clean.loc[
            originations_clean["extreme_ratio_review_flag"],
            "original_cltv"
        ]
        <
        originations_clean.loc[
            originations_clean["extreme_ratio_review_flag"],
            "original_ltv"
        ]
    ).sum()
})

extreme_ratio_check

flagged_loans               3
flagged_harp_loans          3
flagged_with_pre_harp_id    3
flagged_cltv_below_ltv      0
dtype: int64

## Export the audited origination dataset

The cleaned origination table is saved in Parquet format. The raw source files remain
unchanged, and the processed dataset is excluded from Git because it contains
loan-level data.

In [16]:
processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

clean_output_file = (
    processed_data_dir / "originations_clean.parquet"
)

originations_clean.to_parquet(
    clean_output_file,
    index=False
)

export_check = pd.Series({
    "file_created": clean_output_file.exists(),
    "file_size_mb": round(
        clean_output_file.stat().st_size / (1024**2),
        2
    ),
    "exported_rows": len(originations_clean),
    "exported_columns": originations_clean.shape[1]
})

export_check

file_created          True
file_size_mb          3.81
exported_rows       200000
exported_columns        33
dtype: object

In [17]:
originations_reloaded = pd.read_parquet(clean_output_file)

parquet_validation = pd.Series({
    "reloaded_rows": len(originations_reloaded),
    "reloaded_columns": originations_reloaded.shape[1],
    "unique_loan_ids": originations_reloaded[
        "loan_identifier"
    ].nunique(),
    "missing_loan_ids": originations_reloaded[
        "loan_identifier"
    ].isna().sum(),
    "duplicate_loan_ids": originations_reloaded[
        "loan_identifier"
    ].duplicated().sum(),
    "flagged_extreme_ratios": originations_reloaded[
        "extreme_ratio_review_flag"
    ].sum(),
    "column_order_matches": (
        originations_reloaded.columns.tolist()
        == originations_clean.columns.tolist()
    )
})

parquet_validation

reloaded_rows             200000
reloaded_columns              33
unique_loan_ids           200000
missing_loan_ids               0
duplicate_loan_ids             0
flagged_extreme_ratios         3
column_order_matches        True
dtype: object